# Module 7 lab: bounded and honest failure

Fake dependency, clock/sleeper concepts, and no network or real sleep.

## Objectives and predictions

Predict 1: should validation retry? Predict 2: should a repeated idempotency key duplicate an effect? Predict 3: what should an unexpected defect reveal?

In [ ]:
class Transient(Exception): pass
class Permanent(Exception): pass
class Timeout(Exception): pass
class FakeDependency:
 def __init__(self,outcomes): self.outcomes=list(outcomes); self.calls=0; self.effects=0
 def fetch(self,key,deadline):
  self.calls+=1; o=self.outcomes.pop(0) if self.outcomes else "ok"
  if o=="transient": raise Transient()
  if o=="permanent": raise Permanent()
  if o=="timeout": raise Timeout()
  return {"key":key,"value":"source"}
class Sleeper:
 def __init__(self): self.delays=[]
 def sleep(self,d): self.delays.append(d)
def run(dep,sleeper,key,body,store,max_attempts=3):
 rid="req-1"
 if not isinstance(body,dict) or not body.get("title"): return {"status":400,"body":{"error":{"code":"invalid_input","request_id":rid}}}
 fp=(key,body["title"])
 if key in store:
  if store[key][0]!=fp: return {"status":409,"body":{"error":{"code":"idempotency_conflict","request_id":rid}}}
  return store[key][1]
 for attempt in range(1,max_attempts+1):
  try:
   source=dep.fetch(key,deadline=10); dep.effects+=1; result={"status":201,"body":{"title":body["title"],"source":source["value"],"request_id":rid}}; store[key]=(fp,result); return result
  except Transient:
   if attempt==max_attempts: return {"status":503,"body":{"error":{"code":"dependency_unavailable","request_id":rid}}}
   sleeper.sleep(2**(attempt-1))
  except (Permanent,Timeout): return {"status":502,"body":{"error":{"code":"dependency_failed","request_id":rid}}}
  except Exception: return {"status":500,"body":{"error":{"code":"internal_error","request_id":rid}}}
store={}; dep=FakeDependency(["transient","ok"]); sl=Sleeper(); result=run(dep,sl,"k",{"title":"report"},store)
assert result["status"]==201 and dep.calls==2 and sl.delays==[1]; replay=run(dep,sl,"k",{"title":"report"},store); assert replay==result and dep.calls==2 and dep.effects==1

## Prediction answers

1. Validation is not transient, so it is not retried. 2. A repeated key returns the original result without another side effect. 3. An unexpected defect becomes a safe generic error with a request ID, while diagnostics stay internal.

In [ ]:
assert run(FakeDependency(["ok"]),Sleeper(),"bad",{}, {})["status"]==400
assert run(FakeDependency(["permanent"]),Sleeper(),"p",{"title":"x"}, {})["status"]==502
assert run(FakeDependency(["timeout"]),Sleeper(),"t",{"title":"x"}, {})["status"]==502
assert run(FakeDependency(["transient","transient","transient"]),Sleeper(),"e",{"title":"x"}, {})["status"]==503
ai="while True: try work() except Exception: continue; return success"; assert "while True" in ai and "except Exception" in ai
def retryable(e): return isinstance(e,Transient)
assert retryable(Transient()) and not retryable(Permanent()) and not retryable(Timeout())

## AI review, breaker, and guided reasoning

Flag infinite retry, catch-all success, no deadline, raw exception output, and unsafe replay. A circuit breaker opens after repeated failures, then needs a controlled trial; it must not invent success.

In [ ]:
class Breaker:
 def __init__(self): self.state="closed"; self.failures=0
 def record(self,ok):
  if ok: self.state="closed"; self.failures=0
  else:
   self.failures+=1
   if self.failures>=2: self.state="open"
br=Breaker(); br.record(False); br.record(False); assert br.state=="open"; br.record(True); assert br.state=="closed"

## Independent challenge and answers

Add a fake clock and half-open cooldown without sleeping. Answers: permanent failures do not improve by retry; deadline bounds total waiting; timeout after a write is ambiguous; truthful fallback reports stored/deferred work; traces/SQL/secrets belong only in safe internal evidence. This does not prove distributed crash recovery or exactly-once behavior.

Evidence: failure matrix, hypotheses, statuses/bodies/attempts/effects, recorded delays, replay, breaker drill, AI review, clean run.

## Baseline reproduction: slow path
The unsafe baseline catches everything, retries forever, and may report success after dependency failure. Reproduce its categories as a failure matrix, without actually looping or sleeping.

In [ ]:
baseline_matrix=[("validation","retry=no"),("transient","retry=bounded"),("permanent","retry=no"),("timeout","deadline"),("unexpected","safe 500")]
assert len(baseline_matrix)==5
print("baseline matrix recorded:",baseline_matrix)

## Pre-edit hypothesis
Write: (1) a classified retry with a maximum attempt count bounds waiting; (2) an idempotency key makes a replay return the original result without another effect.

In [ ]:
retry_hypothesis="classified transient retry with max attempts bounds waiting"
idempotency_hypothesis="same key and fingerprint replay stored result without a second effect"
assert "max attempts" in retry_hypothesis and "stored result" in idempotency_hypothesis

## Incremental guided implementation: classify first
Classification precedes retry. Validation, permanent failure, timeout, cancellation, and unexpected defects have different handling; only the known transient exception is retryable.

In [ ]:
def classify(exc):
 if isinstance(exc,Transient): return "transient"
 if isinstance(exc,Permanent): return "permanent"
 if isinstance(exc,Timeout): return "timeout"
 return "unexpected"
assert classify(Transient())=="transient" and classify(Permanent())=="permanent" and classify(Timeout())=="timeout"

In [ ]:
def bounded_attempts(dep,sleeper):
 out=run(dep,sleeper,"bounded",{"title":"x"},{},max_attempts=2)
 return out
limited=bounded_attempts(FakeDependency(["transient","transient","transient"]),Sleeper())
assert limited["status"]==503

## Deadlines, idempotency, and truthful failure
A deadline bounds an attempt that hangs; a retry count alone does not. The local run passes a deadline marker to the fake. A replay checks both key and fingerprint.

In [ ]:
same_store={}; same_dep=FakeDependency(["ok"]); same_sleep=Sleeper()
first_result=run(same_dep,same_sleep,"same",{"title":"x"},same_store)
second_result=run(same_dep,same_sleep,"same",{"title":"x"},same_store)
assert first_result==second_result and same_dep.calls==1 and same_dep.effects==1
conflict=run(same_dep,same_sleep,"same",{"title":"different"},same_store)
assert conflict["status"]==409

## Positive, negative, and failure checks
Exercise success/recovery, invalid input, permanent failure, timeout, transient exhaustion, duplicate replay, and unexpected defect. Responses must not expose exception text.

In [ ]:
assert run(FakeDependency(["transient","ok"]),Sleeper(),"r",{"title":"ok"},{})["status"]==201
assert run(FakeDependency(["permanent"]),Sleeper(),"p",{"title":"x"},{})["status"]==502
assert run(FakeDependency(["timeout"]),Sleeper(),"t",{"title":"x"},{})["status"]==502
error_response={"status":500,"body":{"error":{"code":"internal_error","request_id":"req-1"}}}
assert "Traceback" not in str(error_response) and "internal_error" in str(error_response)

## AI-style/broken-code critique
The generated loop retries every exception forever and returns success. A reviewer should ask for attempt/time bounds, classification, cancellation behavior, and replay safety.

In [ ]:
broken_resilience="while True: try work() except Exception: sleep(1); return success"
assert "while True" in broken_resilience and "except Exception" in broken_resilience
print("Reject infinite retry and false success.")

## Guided TODO: attempt
Implement a small breaker that opens after two failures, then closes after a successful trial. A real cooldown would use a clock; this state exercise keeps it deterministic.

In [ ]:
class TodoBreaker:
 def __init__(self): self.state="closed"; self.failures=0
 def record(self,ok):
  if ok: self.state="closed"; self.failures=0
  else:
   self.failures+=1
   if self.failures>=2: self.state="open"
tb=TodoBreaker(); tb.record(False); tb.record(False); assert tb.state=="open"; tb.record(True); assert tb.state=="closed"


## Reference solution
A production breaker needs open/half-open timing and an operator policy. The local reference makes only the state transition claim it tests.

In [ ]:
def reference_breaker(states):
 failures=0
 for ok in states:
  if ok: failures=0
  else: failures+=1
 return "open" if failures>=2 else "closed"
assert reference_breaker([False,False])=="open" and reference_breaker([False,False,True])=="closed"


## Independent challenge
Add a fake clock and allow one half-open trial after a cooldown. Never call real sleep; record requested delays and explain shutdown/cancellation behavior.

In [ ]:
recovery={"fallback":"defer with visible accepted state","shutdown":"stop new work and drain boundedly"}
assert "visible" in recovery["fallback"] and "bounded" in recovery["shutdown"]

## Exit questions and Answers
Permanent errors are not repaired by retry. A deadline bounds an individual wait and total budget. A timeout after a side effect is ambiguous, so idempotency/deduplication is needed. Truthful degradation reports deferred work, not fake success. Traces and secrets belong in safe internal evidence only.

## Evidence handoff
Keep failure matrix, hypotheses, classifications, attempts/delays/deadline assumptions, side-effect counts, duplicate replay, breaker drill, AI critique, TODO/reference, challenge recovery policy, and clean execution. State the crash window you do not prove.